In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

import os
import re
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score, accuracy_score
)
from xgboost import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("[경고] LightGBM 미설치 → LightGBM 모델 사용 불가")

try:
    from imblearn.over_sampling import SMOTE, BorderlineSMOTE
    HAS_IMBLEARN = True
except ImportError:
    HAS_IMBLEARN = False

try:
    from ctgan import CTGAN
    HAS_CTGAN = True
except ImportError:
    HAS_CTGAN = False

warnings.filterwarnings("ignore")


# ============================================================
# 1. 전역 경로 설정
# ============================================================
BASE_DIR         = r'중간결과'
TOP10_FOLDER     = os.path.join(BASE_DIR, r'15_우수모델\Top10')
TRAIN_FOLDER     = os.path.join(BASE_DIR, '12_train')
TEST_FOLDER      = os.path.join(BASE_DIR, '12_test')
FEATURE_FOLDER   = os.path.join(BASE_DIR, '13_피처셀렉션')

# 저장 폴더 분리
SAVE_PD_DIR      = os.path.join(BASE_DIR, r'15_우수모델\2014_2024_PD')
SAVE_SUMMARY_DIR = os.path.join(BASE_DIR, r'15_우수모델\요약')
SAVE_OTHERS_DIR  = os.path.join(BASE_DIR, r'15_우수모델\나머지')

for d in [SAVE_PD_DIR, SAVE_SUMMARY_DIR, SAVE_OTHERS_DIR]:
    os.makedirs(d, exist_ok=True)

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
YEAR_COL     = "회계년도"
ID_COLS      = ["회사명", "사업자등록번호", "회계년도"]
FOLD_VAL_YEARS = [2016, 2017, 2018, 2019, 2020, 2021]
TRAIN_START    = 2012
RECALL_MIN     = 0.9


# ============================================================
# 2. Top10 CSV → 업종코드/폴더명 추출 유틸
# ============================================================

def parse_industry_info(top10_csv_path: str):
    """
    파일명 예시: M03_음식료품_제조업_Top10 우수모델.csv
    → industry_code = "M03"
    → industry_folder = "M03_음식료품_제조업"   (13_피처셀렉션 하위 폴더명)
    → train_file = "M03_음식료품_제조업_train.parquet"
    → test_file  = "M03_음식료품_제조업_test.parquet"
    → prefix     = "M03_음식료품_제조업"
    """
    fname = os.path.basename(top10_csv_path)           # M03_음식료품_제조업_Top10 우수모델.csv
    # "_Top10" 이전 부분을 업종 식별자로 사용
    match = re.match(r'^(.+?)_Top10', fname)
    if not match:
        raise ValueError(f"파일명 패턴 불일치: {fname}")
    prefix = match.group(1)                             # M03_음식료품_제조업
    code   = prefix.split('_')[0]                       # M03

    return {
        "prefix"         : prefix,
        "code"           : code,
        "train_file"     : os.path.join(TRAIN_FOLDER, f"{prefix}_train.parquet"),
        "test_file"      : os.path.join(TEST_FOLDER,  f"{prefix}_test.parquet"),
        "feature_folder" : os.path.join(FEATURE_FOLDER, prefix),
    }


# ============================================================
# 3. 모델 팩토리 — Top10 Model 컬럼값 기반으로 모델 생성
# ============================================================

def make_model(model_name: str, pos_weight: float):
    """
    model_name : "XGBoost" / "LightGBM" / "RandomForest" / "LogisticRegression"
    pos_weight : ClassWeight 방식일 때 양성 클래스 가중치 (그 외 방식은 1.0)
    """
    mn = str(model_name).strip()

    if mn == "XGBoost":
        return XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="aucpr",
            random_state=RANDOM_STATE, verbosity=0,
            scale_pos_weight=pos_weight        # ClassWeight → pos_weight, 그 외 → 1.0
        )

    if mn == "LightGBM":
        if not HAS_LGBM:
            raise ImportError("LightGBM 미설치. pip install lightgbm")
        if pos_weight > 1.0:
            # ClassWeight 방식 → is_unbalance 대신 scale_pos_weight 사용
            return LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, verbose=-1,
                scale_pos_weight=pos_weight
            )
        else:
            return LGBMClassifier(
                n_estimators=300, learning_rate=0.05, max_depth=4,
                subsample=0.8, colsample_bytree=0.8,
                random_state=RANDOM_STATE, verbose=-1
            )

    if mn == "RandomForest":
        if pos_weight > 1.0:
            return RandomForestClassifier(
                n_estimators=300, max_depth=4, min_samples_leaf=5,
                random_state=RANDOM_STATE, n_jobs=-1,
                class_weight="balanced"
            )
        else:
            return RandomForestClassifier(
                n_estimators=300, max_depth=4, min_samples_leaf=5,
                random_state=RANDOM_STATE, n_jobs=-1
            )

    if mn == "LogisticRegression":
        if pos_weight > 1.0:
            return LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                max_iter=1000, random_state=RANDOM_STATE,
                class_weight="balanced"
            )
        else:
            return LogisticRegression(
                penalty="l2", C=1.0, solver="lbfgs",
                max_iter=1000, random_state=RANDOM_STATE
            )

    raise ValueError(f"지원하지 않는 모델명: {model_name}")


# ============================================================
# 4. 불균형 처리 함수
# ============================================================

def _parse_smote_ratio(smote_ratio):
    try:
        if smote_ratio is None:
            return None
        s = str(smote_ratio).strip()
        if s in ("", "-", "nan", "None"):
            return None
        return float(s)
    except (TypeError, ValueError):
        return None


def _ctgan_resample(X, y, ratio):
    if not HAS_CTGAN:
        raise ImportError("CTGAN 방식을 사용하려면 'pip install ctgan'이 필요합니다.")
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)
    minority  = X[y == 1].copy()
    n_minority = len(minority)
    n_majority = (y == 0).sum()
    target_n  = int(n_majority * ratio) if ratio else n_majority
    n_synth   = max(0, target_n - n_minority)
    if n_synth == 0 or n_minority < 5:
        return X, y
    ctgan = CTGAN(epochs=300, verbose=False)
    ctgan.fit(minority)
    synth = ctgan.sample(n_synth)
    synth = synth[X.columns]
    X_res = pd.concat([X, synth], ignore_index=True)
    y_res = pd.concat([y, pd.Series([1] * n_synth)], ignore_index=True)
    return X_res, y_res


def apply_resampling(X, y, method, smote_ratio):
    method_l = str(method).strip().lower()
    ratio    = _parse_smote_ratio(smote_ratio)

    if method_l == "classweight":
        pos_weight = (y == 0).sum() / (y == 1).sum()
        return X, y, pos_weight

    if method_l in ("none", "", "-", "nan"):
        return X, y, 1.0

    if "borderline" in method_l:
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = BorderlineSMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if method_l == "smote":
        if not HAS_IMBLEARN:
            raise ImportError("pip install imbalanced-learn")
        sampler = SMOTE(
            sampling_strategy=ratio if ratio is not None else "auto",
            random_state=RANDOM_STATE,
        )
        X_res, y_res = sampler.fit_resample(X, y)
        return X_res, y_res, 1.0

    if "ctgan" in method_l:
        X_res, y_res = _ctgan_resample(X, y, ratio)
        return X_res, y_res, 1.0

    print(f"  [경고] 알 수 없는 Method='{method}' → ClassWeight로 처리합니다.")
    pos_weight = (y == 0).sum() / (y == 1).sum()
    return X, y, pos_weight


# ============================================================
# 5. 평가 유틸
# ============================================================

def find_threshold_at_recall(y_true, y_prob, recall_min=RECALL_MIN):
    thresholds = np.arange(0.01, 1.0, 0.01)
    valid = []
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        rec = recall_score(y_true, y_pred, zero_division=0)
        if rec >= recall_min:
            valid.append(round(thr, 2))
    return max(valid) if valid else None


def calc_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "ROC_AUC"   : roc_auc_score(y_true, y_prob),
        "PR_AUC"    : average_precision_score(y_true, y_prob),
        "Accuracy"  : accuracy_score(y_true, y_pred),
    }


# ============================================================
# 6. 시각화 함수
# ============================================================

plt.rcParams.update({
    "font.family"       : "DejaVu Sans",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.color"        : "#E5E5E5",
    "grid.linewidth"    : 0.7,
    "axes.facecolor"    : "#FAFAFA",
    "figure.facecolor"  : "white",
})

COLOR_NEG  = "#2F6EBA"
COLOR_POS  = "#D94F3D"
COLOR_THR  = "#F5A623"
ALPHA_HIST = 0.72
BINS       = 45


def plot_dist(ax, y_true, y_prob, title, n_total, threshold):
    arr0 = y_prob[np.array(y_true) == 0]
    arr1 = y_prob[np.array(y_true) == 1]
    counts0, edges0 = np.histogram(arr0, bins=BINS, range=(0, 1))
    counts1, edges1 = np.histogram(arr1, bins=BINS, range=(0, 1))
    ax.bar(edges0[:-1], counts0, width=np.diff(edges0),
           align="edge", color=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)", zorder=3)
    ax.bar(edges1[:-1], counts1, width=np.diff(edges1),
           align="edge", color=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)", zorder=3)
    ax.axvline(threshold, color=COLOR_THR, linestyle="--",
               linewidth=1.8, zorder=5, label=f"Threshold = {threshold:.2f}")
    ax.axvspan(threshold, 1.0, alpha=0.06, color=COLOR_POS, zorder=2)
    n0, n1 = len(arr0), len(arr1)
    ir = n1 / n0 if n0 > 0 else float("nan")
    above_thr = (y_prob >= threshold).sum()
    stats_txt = (
        f"N={n_total:,}  |  Normal={n0:,}  Distress={n1:,}\n"
        f"Imbalance ratio = {ir:.3f}  |  Predicted Positive = {above_thr:,}"
    )
    ax.text(0.98, 0.97, stats_txt, transform=ax.transAxes, fontsize=8.2,
            va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="#CCCCCC", alpha=0.85))
    ax.set_title(title, fontsize=12, fontweight="bold", pad=10)
    ax.set_xlabel("Predicted Probability", fontsize=9.5)
    ax.set_ylabel("Count", fontsize=9.5)
    ax.set_xlim(0, 1)
    ax.tick_params(labelsize=8.5)
    legend_elems = [
        Patch(facecolor=COLOR_NEG, alpha=ALPHA_HIST, label="Normal (0)"),
        Patch(facecolor=COLOR_POS, alpha=ALPHA_HIST, label="Distress (1)"),
        Line2D([0], [0], color=COLOR_THR, linestyle="--", linewidth=1.8,
               label=f"Threshold = {threshold:.2f}"),
    ]
    ax.legend(handles=legend_elems, fontsize=8.5, framealpha=0.9,
              loc="upper left", edgecolor="#CCCCCC")


def save_prob_dist_plot(val_prob_df, y_test, y_prob_test, threshold,
                        method, feature_set, model_name, prefix):
    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(
        2, 2, height_ratios=[3.2, 1],
        hspace=0.42, wspace=0.32,
        left=0.07, right=0.97, top=0.91, bottom=0.05
    )
    ax_cv   = fig.add_subplot(gs[0, 0])
    ax_test = fig.add_subplot(gs[0, 1])
    ax_tbl  = fig.add_subplot(gs[1, :])
    ax_tbl.axis("off")

    plot_dist(ax_cv, val_prob_df["y_true"].values, val_prob_df["y_prob"].values,
              "Expanding Window CV — Predicted Probability Distribution",
              n_total=len(val_prob_df), threshold=threshold)
    plot_dist(ax_test, y_test.values, y_prob_test,
              "Hold-out Test — Predicted Probability Distribution",
              n_total=len(y_test), threshold=threshold)

    # 테이블
    cv_mean_      = val_prob_df.groupby("Val_Year").apply(
        lambda g: pd.Series(calc_metrics(g["y_true"], g["y_prob"], threshold))
    ).mean()
    test_metrics_ = calc_metrics(y_test, y_prob_test, threshold)
    metrics_order = ["F1", "Recall", "Precision", "ROC_AUC", "PR_AUC", "Accuracy"]
    col_labels    = ["Split"] + metrics_order
    cv_row   = ["CV Mean"] + [f"{float(cv_mean_[m]):.4f}" for m in metrics_order]
    test_row = ["Test"]    + [f"{test_metrics_[m]:.4f}"   for m in metrics_order]
    gap_row  = ["Gap (Test − CV)"] + [
        f"{test_metrics_[m] - float(cv_mean_[m]):+.4f}" for m in metrics_order
    ]
    table_data = [cv_row, test_row, gap_row]
    tbl = ax_tbl.table(cellText=table_data, colLabels=col_labels,
                        cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.scale(1, 1.7)
    for j in range(len(col_labels)):
        tbl[(0, j)].set_facecolor("#2F4F7F")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    row_colors = ["#EEF3FA", "#FAFAFA", "#FFF4EE"]
    for i, rc in enumerate(row_colors, start=1):
        for j in range(len(col_labels)):
            tbl[(i, j)].set_facecolor(rc)
    for j, m in enumerate(metrics_order, start=1):
        gap_val = test_metrics_[m] - float(cv_mean_[m])
        color   = "#C0392B" if gap_val < -0.02 else ("#27AE60" if gap_val > 0.02 else "#555555")
        tbl[(3, j)].set_text_props(color=color, fontweight="bold")
    ax_tbl.set_title("Performance Summary", fontsize=10, fontweight="bold", pad=6, loc="left")

    fig.suptitle(
        f"{model_name} | {method} | {feature_set} | {prefix} | "
        f"Threshold(Recall≥{RECALL_MIN}) = {threshold:.2f}",
        fontsize=13.5, fontweight="bold", y=0.975
    )
    save_path = os.path.join(SAVE_OTHERS_DIR, f"{prefix}_{feature_set}_prob_distribution.png")
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.close()
    return save_path


# ============================================================
# 7. 단일 업종 처리 함수
# ============================================================

def process_one_industry(top10_csv_path: str):
    info   = parse_industry_info(top10_csv_path)
    prefix = info["prefix"]

    print(f"\n{'='*70}")
    print(f"[업종] {prefix}")
    print(f"{'='*70}")

    # ── Top10 1위 행 로드
    top10 = pd.read_csv(top10_csv_path, index_col=0)
    if len(top10) == 0:
        print(f"  [경고] {top10_csv_path} 가 비어있습니다. skip.")
        return

    row          = top10.iloc[0]
    FEATURE_SET  = row["FeatureSet"]
    FEATURE_FILE = row["FeatureFile"]
    METHOD       = row["Method"]
    SMOTE_RATIO  = row["SMOTE_Ratio"]
    MODEL_NAME   = row["Model"]

    print(f"  FeatureSet  : {FEATURE_SET}")
    print(f"  FeatureFile : {FEATURE_FILE}")
    print(f"  Method      : {METHOD}")
    print(f"  SMOTE_Ratio : {SMOTE_RATIO}")
    print(f"  Model       : {MODEL_NAME}")

    # ── 피처 파일 경로 구성
    # FEATURE_FILE 예: "lasso_features_top50--41.csv"
    # 경로 예: 중간결과\13_피처셀렉션\M03_음식료품_제조업\lasso_features_top50--41.csv
    feature_path = os.path.join(info["feature_folder"], FEATURE_FILE)
    if not os.path.exists(feature_path):
        print(f"  [경고] 피처 파일 없음: {feature_path} → skip")
        return

    # ── 데이터 로드
    if not os.path.exists(info["train_file"]):
        print(f"  [경고] train 파일 없음: {info['train_file']} → skip")
        return
    if not os.path.exists(info["test_file"]):
        print(f"  [경고] test 파일 없음: {info['test_file']} → skip")
        return

    train_full = pd.read_parquet(info["train_file"])
    test       = pd.read_parquet(info["test_file"])
    y_train_full = train_full[TARGET_COL]
    y_test       = test[TARGET_COL]

    feat_df      = pd.read_csv(feature_path)
    col_key      = "feature" if "feature" in feat_df.columns else feat_df.columns[0]
    use_features = [f for f in feat_df[col_key].tolist() if f in train_full.columns]

    print(f"\n  Train shape : {train_full.shape} | Test shape : {test.shape}")
    print(f"  피처 수      : {len(use_features)}개  | 양성비율(train): "
          f"{y_train_full.mean()*100:.2f}%")

    # ── Expanding Window CV
    print(f"\n  ── Expanding Window CV")
    fold_results = []
    for val_year in FOLD_VAL_YEARS:
        train_idx = train_full.index[
            (train_full[YEAR_COL] >= TRAIN_START) & (train_full[YEAR_COL] < val_year)
        ]
        val_idx = train_full.index[train_full[YEAR_COL] == val_year]
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue

        X_ft = train_full.loc[train_idx, use_features]
        y_ft = train_full.loc[train_idx, TARGET_COL]
        X_fv = train_full.loc[val_idx,   use_features]
        y_fv = train_full.loc[val_idx,   TARGET_COL]

        imp = SimpleImputer(strategy="median")
        X_ft = pd.DataFrame(imp.fit_transform(X_ft), columns=use_features)
        X_fv = pd.DataFrame(imp.transform(X_fv),     columns=use_features)

        X_res, y_res, pw = apply_resampling(X_ft, y_ft, METHOD, SMOTE_RATIO)
        model = make_model(MODEL_NAME, pw)
        model.fit(X_res, y_res)
        y_prob_val = model.predict_proba(X_fv)[:, 1]

        print(f"    Val {val_year} | ROC_AUC={roc_auc_score(y_fv, y_prob_val):.4f} "
              f"PR_AUC={average_precision_score(y_fv, y_prob_val):.4f}")
        fold_results.append({
            "Val_Year": val_year,
            "y_true"  : y_fv.values,
            "y_prob"  : y_prob_val
        })

    # ── 최종 모델 학습 (전체 train)
    imputer_final = SimpleImputer(strategy="median")
    X_train_all   = pd.DataFrame(
        imputer_final.fit_transform(train_full[use_features]), columns=use_features
    )
    X_test_imp    = pd.DataFrame(
        imputer_final.transform(test[use_features]), columns=use_features
    )
    X_train_res, y_train_res, final_pw = apply_resampling(
        X_train_all, y_train_full, METHOD, SMOTE_RATIO
    )
    final_model = make_model(MODEL_NAME, final_pw)
    final_model.fit(X_train_res, y_train_res)
    y_prob_test = final_model.predict_proba(X_test_imp)[:, 1]

    # ── Threshold 탐색 (Test 기준, Recall >= RECALL_MIN)
    found_thr = find_threshold_at_recall(y_test, y_prob_test, RECALL_MIN)
    THRESHOLD = found_thr if found_thr is not None else 0.01
    if found_thr is None:
        print(f"  [경고] Recall>={RECALL_MIN} 만족하는 threshold 없음 → 0.01 사용")

    test_metrics = calc_metrics(y_test, y_prob_test, THRESHOLD)
    print(f"\n  Test threshold = {THRESHOLD:.2f} | "
          f"Recall={test_metrics['Recall']:.4f} "
          f"Precision={test_metrics['Precision']:.4f} "
          f"F1={test_metrics['F1']:.4f}")

    # ── CV fold별 성능 (Test threshold 적용)
    cv_rows      = []
    val_prob_all = []
    for fr in fold_results:
        m = calc_metrics(fr["y_true"], fr["y_prob"], THRESHOLD)
        cv_rows.append({"Val_Year": fr["Val_Year"], **m})
        val_prob_all.append(pd.DataFrame({
            "Val_Year": fr["Val_Year"],
            "y_true"  : fr["y_true"],
            "y_prob"  : fr["y_prob"],
            "y_pred"  : (fr["y_prob"] >= THRESHOLD).astype(int),
        }))

    cv_df       = pd.DataFrame(cv_rows).round(4)
    cv_mean     = cv_df[["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]].mean()
    val_prob_df = pd.concat(val_prob_all, ignore_index=True) if val_prob_all else pd.DataFrame()

    # ── 파일명 공통 접두사
    file_prefix = f"{prefix}_{FEATURE_SET}"

    # ────────────────────────────────────────────────
    # 저장 ①: 요약 (summary)
    # ────────────────────────────────────────────────
    summary = {
        "Industry"         : prefix,
        "FeatureSet"       : FEATURE_SET,
        "FeatureFile"      : FEATURE_FILE,
        "N_Features"       : len(use_features),
        "Method"           : METHOD,
        "SMOTE_Ratio"      : SMOTE_RATIO,
        "Model"            : MODEL_NAME,
        "Recall_min"       : RECALL_MIN,
        "Threshold_basis"  : "Test",
        "Threshold"        : THRESHOLD,
    }
    for col in ["F1","Recall","Precision","ROC_AUC","PR_AUC","Accuracy"]:
        summary[f"CV_Val_{col}"] = round(float(cv_mean[col]), 4)
        summary[f"Test_{col}"]   = round(test_metrics[col], 4)
        summary[f"Gap_{col}"]    = round(test_metrics[col] - float(cv_mean[col]), 4)

    pd.DataFrame([summary]).to_csv(
        os.path.join(SAVE_SUMMARY_DIR, f"{file_prefix}_summary.csv"),
        index=False, encoding="utf-8-sig"
    )

    # ────────────────────────────────────────────────
    # 저장 ②: 나머지 (CV 결과, 확률분포 이미지, test predictions)
    # ────────────────────────────────────────────────
    cv_df.to_csv(
        os.path.join(SAVE_OTHERS_DIR, f"{file_prefix}_CV_results.csv"),
        index=False, encoding="utf-8-sig"
    )

    y_pred_test = (y_prob_test >= THRESHOLD).astype(int)
    test_id_cols = [c for c in ID_COLS if c in test.columns]
    test_pred_df = test[test_id_cols].copy().reset_index(drop=True)
    test_pred_df["y_true"]  = y_test.values
    test_pred_df["y_prob"]  = y_prob_test.round(4)
    test_pred_df["y_pred"]  = y_pred_test
    test_pred_df["correct"] = (test_pred_df["y_true"] == test_pred_df["y_pred"]).astype(int)
    test_pred_df["error_type"] = "TN"
    test_pred_df.loc[(test_pred_df["y_true"]==1)&(test_pred_df["y_pred"]==1), "error_type"] = "TP"
    test_pred_df.loc[(test_pred_df["y_true"]==1)&(test_pred_df["y_pred"]==0), "error_type"] = "FN"
    test_pred_df.loc[(test_pred_df["y_true"]==0)&(test_pred_df["y_pred"]==1), "error_type"] = "FP"
    test_pred_df.to_csv(
        os.path.join(SAVE_OTHERS_DIR, f"{file_prefix}_test_predictions.csv"),
        index=False, encoding="utf-8-sig"
    )

    if not val_prob_df.empty:
        img_path = save_prob_dist_plot(
            val_prob_df, y_test, y_prob_test, THRESHOLD,
            METHOD, FEATURE_SET, MODEL_NAME, prefix
        )
        print(f"  확률분포 이미지 → {img_path}")

    # ────────────────────────────────────────────────
    # 저장 ③: 2014~2024 PD 데이터
    # ────────────────────────────────────────────────
    PD_YEARS = list(range(2014, 2025))
    all_data = pd.concat([train_full, test], ignore_index=True)
    ady      = all_data[all_data[YEAR_COL].isin(PD_YEARS)].copy()

    X_ady_imp = pd.DataFrame(
        imputer_final.transform(ady[use_features]),
        columns=use_features, index=ady.index
    )
    ady["y_prob"]  = final_model.predict_proba(X_ady_imp)[:, 1].round(4)
    ady["y_pred"]  = (ady["y_prob"] >= THRESHOLD).astype(int)
    ady["y_true"]  = ady[TARGET_COL]
    ady["correct"] = (ady["y_true"] == ady["y_pred"]).astype(int)
    ady["error_type"] = "TN"
    ady.loc[(ady["y_true"]==1)&(ady["y_pred"]==1), "error_type"] = "TP"
    ady.loc[(ady["y_true"]==1)&(ady["y_pred"]==0), "error_type"] = "FN"
    ady.loc[(ady["y_true"]==0)&(ady["y_pred"]==1), "error_type"] = "FP"

    id_cols_avail = [c for c in ID_COLS if c in ady.columns]
    pd_df = ady[id_cols_avail + ["y_true","y_prob","y_pred","correct","error_type"]] \
        .sort_values([id_cols_avail[1], YEAR_COL]).reset_index(drop=True)

    pd_save_path = os.path.join(SAVE_PD_DIR, f"{file_prefix}_2014_2024_PD_데이터.csv")
    pd_df.to_csv(pd_save_path, index=False, encoding="utf-8-sig")
    print(f"  PD 데이터(2014~2024) → {pd_save_path}  ({len(pd_df)}행)")
    print(f"  error_type 분포:\n{pd_df['error_type'].value_counts().to_string()}")

    return summary


# ============================================================
# 8. 전체 Top10 파일 순회 실행
# ============================================================

def main():
    top10_files = sorted(glob.glob(os.path.join(TOP10_FOLDER, "*.csv")))
    if not top10_files:
        print(f"[오류] Top10 파일이 없습니다: {TOP10_FOLDER}")
        return

    print(f"처리할 업종 수: {len(top10_files)}개")
    for f in top10_files:
        print(f"  {os.path.basename(f)}")

    all_summaries = []
    failed = []

    for top10_csv in top10_files:
        try:
            result = process_one_industry(top10_csv)
            if result:
                all_summaries.append(result)
        except Exception as e:
            industry_name = os.path.basename(top10_csv)
            print(f"\n  [오류] {industry_name} 처리 중 예외 발생: {e}")
            failed.append({"file": industry_name, "error": str(e)})

    # ── 전체 요약 통합 CSV
    if all_summaries:
        all_summary_df = pd.DataFrame(all_summaries)
        all_summary_df.to_csv(
            os.path.join(SAVE_SUMMARY_DIR, "전체_업종_통합_summary.csv"),
            index=False, encoding="utf-8-sig"
        )
        print(f"\n\n{'='*70}")
        print(f"전체 처리 완료: 성공 {len(all_summaries)}개 / 실패 {len(failed)}개")
        print(f"통합 요약 → {os.path.join(SAVE_SUMMARY_DIR, '전체_업종_통합_summary.csv')}")
        print(f"{'='*70}")

    if failed:
        print(f"\n[실패 목록]")
        for f in failed:
            print(f"  {f['file']} : {f['error']}")
        pd.DataFrame(failed).to_csv(
            os.path.join(SAVE_OTHERS_DIR, "실패_목록.csv"),
            index=False, encoding="utf-8-sig"
        )


if __name__ == "__main__":
    main()